# 06. Copying

**Tier:** Intermediate. **Why you'd use this:** A seating chart is a list of row lists. You make what you think is a backup, change one seat, and the "backup" changes too — unless you copied at the right depth.

**After this lesson you can:** tell a shared list from a copy; choose a shallow copy or `deepcopy`; say which nested list an edit will change after each kind of copy.
About 25 minutes. **Uses:** [Lists](01-lists-lesson.ipynb), [Dicts](03-dicts-lesson.ipynb)


## Quick review

1. What does `zip(names, scores)` give you in a `for` loop?
2. When should you pass `strict=True` to `zip`?

<details><summary>Answers</summary>

Pairs `(name, score)` until one list ends. When unequal lengths are a bug you want to catch.
</details>


## The idea

Assigning a list to a new name does not copy it. Both names point at one list. `list(grid)` and `grid[:]` copy the outer list only. The inner lists are still shared. `copy.deepcopy` copies the nest, so an edit to a copied row stays in the copy.

<details><summary>Explain more</summary>

`copy.copy` is the shallow copy: a new outer container, the same inner objects. A slice of a list of numbers is enough when the items themselves cannot contain further lists. A slice of a list of lists is not.
</details>


## See it work

Hand a second class its own grade grid, so marking one class cannot change the other.


In [1]:
import copy


def fork_grid(grid):
    # 1. New outer list and new row lists. Numbers inside need no further copy.
    return copy.deepcopy(grid)


morning = [[1, 2], [3, 4]]
afternoon = fork_grid(morning)
afternoon[0].append(9)
print(morning)
print(afternoon)

[[1, 2], [3, 4]]
[[1, 2, 9], [3, 4]]


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `backup = grid` see the append when `backup` was never passed to `append`?
2. When is `row[:]` enough, and when do you still need `deepcopy`?

<details><summary>Answers</summary>

`backup` and `grid` are two names for one list. A slice is enough when the row holds numbers or strings. If a row holds further lists, the slice still shares those.
</details>


In [2]:
row = [0]
grid = [row, row]
backup = grid
grid[0].append(1)
print(backup)
print(grid[0] is grid[1])

[[0, 1], [0, 1]]
True


<details><summary>Check your answers</summary>

`backup` sees `[1]` in both rows. `grid[0] is grid[1]` is true: `[row, row]` stored one list twice. `backup = grid` added a third name for that same outer list.
</details>


## Why it works this way

Assignment stores a pointer. It does not walk the list. `backup = grid` is O(1) and shares every nested object. `list(grid)` and `grid[:]` allocate a new outer list and copy the pointers that were in it. That is O(n) in the outer length, and the row objects are the same objects. `copy.copy` does this shallow copy for lists and for dicts: a new dict, the same values.

`deepcopy` walks the nest. At each list or dict it allocates a new container and copies into it. It memos objects it has already copied, keyed by identity, not by equality. Two separate lists that happen to hold the same numbers are copied twice. One list stored in two slots is one object, so it is copied once and both slots of the copy point at that new row. Equality would glue those equal-but-distinct rows together. The memo also stops a cycle: a list that contains itself is copied once, and the copy points at the copy, instead of walking forever. The cost follows the whole nest. A shallow copy of a dict of lists shares those lists, the same way a shallow grid shares its rows. Only the outer container is new. Sharing stays the default because it is cheap and because many names are supposed to see one object. A shallow copy is the wrong protection when the part you care about is nested. Deep dive: [Mutability and aliasing](../../internals/02-mutability/01-aliasing-lesson.ipynb).


In [3]:
grid = [[0], [0]]
shallow = list(grid)
deep = copy.deepcopy(grid)
print(shallow[0] is grid[0])
print(deep[0] is grid[0])
print(id(shallow[0]) == id(grid[0]))

True
False
True


## When to use it

Use assignment when two names should see one list. Use `grid[:]` or `list(grid)` when the outer list must grow separately and the items are numbers or strings. Use `deepcopy` when the nest must be independent, including dicts whose values are lists. Don't use `deepcopy` on a huge tree you only needed to append to at the top. The copy spends time and memory on every nested object.


## Common mistake

A shallow copy, then an edit to a row. `list(grid)` made a new outer list. The row objects were copied as pointers. Appending to `copied[0]` appends to `grid[0]`.


In [4]:
grid = [[0, 0], [0, 0]]
copied = list(grid)
copied[0].append(1)
print(grid)
print(copied[0] is grid[0])

[[0, 0, 1], [0, 0]]
True


`deepcopy` allocates new rows. The `is` test is false, and the original grid does not gain the `1`.


In [5]:
grid = [[0, 0], [0, 0]]
copied = copy.deepcopy(grid)
copied[0].append(1)
print(grid)
print(copied[0] is grid[0])

[[0, 0], [0, 0]]
False


## Try it

Change `fork_grid` so only the outer list is new and the rows stay shared. Which call replaces `deepcopy`, and what does `afternoon[0].append(9)` do to `morning`?


## Recap

- Assignment shares the object. A slice or `list()` copies the outer pointers only.
- `deepcopy` copies the nest and remembers objects it has already copied.
- A shallow copy still lets an inner append show up on the original.

Next: [practice](06-copying-practice.ipynb).
